# Обучение модели на корпусе данных SBERquad

В данной работе была обучена модель multilingual-e5-small на корпусе данных SBERquad двумя стратегиями:
1) Baseline - стандартная форма обучения в 3 эпохи
2) Curriculum - обучение с варьированием сложности обучающих примеров

Импортируем библиотеки:

In [2]:
import pandas as pd
import numpy as np
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm

In [14]:
import torch
import transformers
from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
from torch.utils.data import DataLoader

Загружаем корпус данных:

In [3]:
dataset = load_dataset("kuznetsoffandrey/sberquad", split="train") 
print(f" Загружено {len(dataset)} примеров")

README.md: 0.00B [00:00, ?B/s]

C:\Users\Александр\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Александр\.cache\huggingface\hub\datasets--kuznetsoffandrey--sberquad. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installe

train-00000-of-00001.parquet:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


validation-00000-of-00001.parquet:   0%|          | 0.00/3.43M [00:00<?, ?B/s]

...

test-00000-of-00001.parquet:   0%|          | 0.00/4.93M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/45328 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5036 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/23936 [00:00<?, ? examples/s]

 Загружено 45328 примеров


In [8]:
all_examples = []

for i, row in enumerate(tqdm(dataset)):
    q = row['question']
    p = row['context']
    id_ = str(row['id'])
    
    if len(str(q)) > 5 and len(str(p)) > 20:
        all_examples.append({
            'query_id': id_,
            'query': q,
            'doc_id': f"doc_{id_}",
            'passage': p
        })

df_full = pd.DataFrame(all_examples)
print(f"Всего валидных пар: {len(df_full)}")

  0%|          | 0/45328 [00:00<?, ?it/s]

Всего валидных пар: 45328


In [9]:
train_df, test_df = train_test_split(df_full, test_size=0.2, random_state=42)

In [10]:
def calculate_difficulty(row):
    q_text = str(row['query']).lower()
    p_text = str(row['passage']).lower()
    
    query_words = set(q_text.split())
    passage_words = set(p_text.split())
    
    intersection = len(query_words & passage_words)
    union = len(query_words | passage_words)
    lexical_overlap = intersection / union if union > 0 else 0
    
    passage_len = len(p_text.split())
    query_len = len(q_text.split())
    
    norm_passage_len = min(passage_len / 300, 1.0) 
    norm_query_len = min(query_len / 50, 1.0)
    
    difficulty = (norm_passage_len * 0.4 + 
                  norm_query_len * 0.2 + 
                  (1 - lexical_overlap) * 0.4)
    
    return difficulty

In [16]:
tqdm.pandas()
train_df['difficulty'] = train_df.apply(calculate_difficulty, axis=1)

In [17]:
train_df.to_csv('sberquad_train.csv', index=False, encoding='utf-8')
test_df.to_csv('sberquad_test.csv', index=False, encoding='utf-8')

In [18]:
df_train = pd.read_csv('sberquad_train.csv')

In [19]:
df_sorted = df_train.sort_values('difficulty').reset_index(drop=True)
n_stages = 4
stage_size = len(df_sorted) // n_stages
stages = {}

In [20]:
for i in range(n_stages):
    start = i * stage_size
    end = (i + 1) * stage_size if i < n_stages - 1 else len(df_sorted)
    name = f'stage_{i+1}'
    stages[name] = df_sorted.iloc[start:end]
    print(f"  {name}: {len(stages[name])} примеров (Avg Diff: {stages[name]['difficulty'].mean():.3f})")

  stage_1: 9065 примеров (Avg Diff: 0.493)
  stage_2: 9065 примеров (Avg Diff: 0.520)
  stage_3: 9065 примеров (Avg Diff: 0.548)
  stage_4: 9067 примеров (Avg Diff: 0.614)


In [21]:
model_name = 'intfloat/multilingual-e5-small'
batch_size = 8
num_epochs_base = 3
num_epochs_curr = 2

In [22]:
val_sample = df_train.sample(n=300, random_state=42)
evaluator = EmbeddingSimilarityEvaluator(
    ["query: " + q for q in val_sample['query']],
    ["passage: " + p for p in val_sample['passage']],
    [1.0] * len(val_sample), name='train-val'
)

In [24]:
from datetime import datetime

In [25]:
print("\n ОБУЧЕНИЕ BASELINE...")
base_model = SentenceTransformer(model_name)
train_loss = losses.MultipleNegativesRankingLoss(base_model)

train_examples = [InputExample(texts=[f"query: {r['query']}", f"passage: {r['passage']}"]) for _, r in df_train.iterrows()]
train_loader = DataLoader(train_examples, shuffle=True, batch_size=batch_size)

base_path = f'models/sberquad_baseline_{datetime.now().strftime("%H%M")}'
base_model.fit(train_objectives=[(train_loader, train_loss)], epochs=num_epochs_base, evaluator=evaluator, output_path=base_path, show_progress_bar=True)


 ОБУЧЕНИЕ BASELINE...


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

C:\Users\Александр\AppData\Local\Programs\Python\Python313\Lib\site-packages\sentence_transformers\evaluation\EmbeddingSimilarityEvaluator.py:195: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  eval_pearson, _ = pearsonr(labels, scores)
C:\Users\Александр\AppData\Local\Programs\Python\Python313\Lib\site-packages\sentence_transformers\evaluation\EmbeddingSimilarityEvaluator.py:196: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  eval_spearman, _ = spearmanr(labels, scores)


In [27]:
import os

In [28]:
print("\n ОБУЧЕНИЕ CURRICULUM...")
curr_model = SentenceTransformer(model_name)
curr_loss = losses.MultipleNegativesRankingLoss(curr_model)
curr_path = f'models/sberquad_curriculum_{datetime.now().strftime("%H%M")}'

for name, stage_df in stages.items():
    print(f"--- {name} ---")
    st_examples = [InputExample(texts=[f"query: {r['query']}", f"passage: {r['passage']}"]) for _, r in stage_df.iterrows()]
    st_loader = DataLoader(st_examples, shuffle=True, batch_size=batch_size)
    curr_model.fit(train_objectives=[(st_loader, curr_loss)], epochs=num_epochs_curr, evaluator=evaluator, output_path=os.path.join(curr_path, name), show_progress_bar=True)

final_path = os.path.join(curr_path, "final_model")
curr_model.save(final_path)
print(f" Финальная модель: {final_path}")


 ОБУЧЕНИЕ CURRICULUM...
--- stage_1 ---


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

C:\Users\Александр\AppData\Local\Programs\Python\Python313\Lib\site-packages\sentence_transformers\evaluation\EmbeddingSimilarityEvaluator.py:195: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  eval_pearson, _ = pearsonr(labels, scores)
C:\Users\Александр\AppData\Local\Programs\Python\Python313\Lib\site-packages\sentence_transformers\evaluation\EmbeddingSimilarityEvaluator.py:196: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  eval_spearman, _ = spearmanr(labels, scores)


--- stage_2 ---


C:\Users\Александр\AppData\Local\Programs\Python\Python313\Lib\site-packages\sentence_transformers\evaluation\EmbeddingSimilarityEvaluator.py:195: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  eval_pearson, _ = pearsonr(labels, scores)
C:\Users\Александр\AppData\Local\Programs\Python\Python313\Lib\site-packages\sentence_transformers\evaluation\EmbeddingSimilarityEvaluator.py:196: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  eval_spearman, _ = spearmanr(labels, scores)


--- stage_3 ---


C:\Users\Александр\AppData\Local\Programs\Python\Python313\Lib\site-packages\sentence_transformers\evaluation\EmbeddingSimilarityEvaluator.py:195: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  eval_pearson, _ = pearsonr(labels, scores)
C:\Users\Александр\AppData\Local\Programs\Python\Python313\Lib\site-packages\sentence_transformers\evaluation\EmbeddingSimilarityEvaluator.py:196: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  eval_spearman, _ = spearmanr(labels, scores)


--- stage_4 ---


C:\Users\Александр\AppData\Local\Programs\Python\Python313\Lib\site-packages\sentence_transformers\evaluation\EmbeddingSimilarityEvaluator.py:195: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  eval_pearson, _ = pearsonr(labels, scores)
C:\Users\Александр\AppData\Local\Programs\Python\Python313\Lib\site-packages\sentence_transformers\evaluation\EmbeddingSimilarityEvaluator.py:196: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  eval_spearman, _ = spearmanr(labels, scores)


 Финальная модель: models/sberquad_curriculum_1032\final_model
